# Learned representations

This notebook regenerates both representation composites from live, sample-level runs. The default `reduced` profile uses compact, class-balanced fixtures curated from genuine N-MNIST and MNE auditory/visual EEG samples. It trains the repository's actual `TemporalMCSNN`/`MCSNN` implementations and extracts second-convolution-layer spikes; no event histogram or proxy embedding is used.

The reduced scores validate the representation mechanism and plotting path. They are not replacements for the longer published sweep. Figures always display inline; saving remains opt-in.

## Configuration

`reduced` and `smoke` run live and offline from committed samples. `publication` replays a strict optional feature archive when supplied, otherwise it shows the read-only reference targets. Set `MNN_USE_REPRESENTATION_ARCHIVE=1` to require that archive rather than falling back.


In [ ]:
from pathlib import Path
from IPython.display import Image, display
import hashlib, io, json, os, platform, shutil, time
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
RUN_PROFILE = os.getenv("MNN_RUN_PROFILE", "reduced")  # reduced | publication | smoke
DEVICE = os.getenv("MNN_DEVICE", "auto")
WORKERS = os.getenv("MNN_WORKERS", "auto")
SAVE_FIGURES = os.getenv("MNN_SAVE_FIGURES", "0") == "1"
OUTPUT_DIR = Path(os.getenv("MNN_OUTPUT_DIR", str(ROOT / "experiments" / "generated_figures")))
OVERWRITE = os.getenv("MNN_OVERWRITE", "0") == "1"
RUN_EXTERNAL_DATA = os.getenv("MNN_RUN_EXTERNAL_DATA", "0") == "1"
ALLOW_DATA_DOWNLOADS = os.getenv("MNN_ALLOW_DATA_DOWNLOADS", "0") == "1"
USE_REPRESENTATION_ARCHIVE = os.getenv("MNN_USE_REPRESENTATION_ARCHIVE", "0") == "1"
DOWNLOAD_REPRESENTATION_DATA = os.getenv("MNN_DOWNLOAD_REPRESENTATION_DATA", "0") == "1"
assert RUN_PROFILE in {"reduced", "publication", "smoke"}

try:
    import torch
except ImportError:
    torch = None
if RUN_PROFILE in {"reduced", "smoke"} and torch is None:
    raise RuntimeError("The reduced and smoke profiles require the repository's Torch environment.")
if DEVICE == "auto" and torch is not None:
    DEVICE_RESOLVED = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
else:
    DEVICE_RESOLVED = DEVICE if DEVICE != "auto" else "cpu"

MANIFEST = json.loads((ROOT / "experiments" / "figure_manifest.json").read_text())
SPEC = {row["filename"]: row for row in MANIFEST["figures"]}
FIGURE_REPORT = []

def _hash(value):
    if value is None:
        return None
    if isinstance(value, Path):
        return hashlib.sha256(value.read_bytes()).hexdigest() if value.exists() else None
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()
    return hashlib.sha256(payload).hexdigest()

def _guard_save(name, provenance):
    if provenance in {"placeholder", "reference-only", "external-gated"}:
        raise RuntimeError(f"Refusing to save non-claimable figure: {name}")
    if "manuscript" in str(OUTPUT_DIR).lower() and not OVERWRITE:
        raise RuntimeError("Writing into a manuscript directory requires MNN_OVERWRITE=1")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    target = OUTPUT_DIR / name
    if target.exists() and not OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite {target}")
    return target

def finish(fig, name, data_source=None, reference=None, seeds=(), *, provenance_class=None, claim_status=None, dpi=220):
    spec = dict(SPEC[name])
    if provenance_class is not None:
        spec["provenance_class"] = provenance_class
    if claim_status is not None:
        spec["claim_status"] = claim_status
    display(fig)
    saved = None
    if SAVE_FIGURES:
        saved = _guard_save(name, spec["provenance_class"])
        fig.savefig(saved, dpi=dpi, bbox_inches="tight", facecolor="white")
    FIGURE_REPORT.append({**spec, "profile": RUN_PROFILE, "device": DEVICE_RESOLVED,
        "seeds": list(seeds), "data_sha256": _hash(data_source),
        "reference_sha256": _hash(reference), "saved_path": str(saved) if saved else None})
    plt.close(fig)

def show_reference_only(name, reason):
    target = ROOT / "experiments" / "assets" / "references" / name
    display(Image(filename=str(target)))
    FIGURE_REPORT.append({**SPEC[name], "profile": RUN_PROFILE, "device": DEVICE_RESOLVED,
        "seeds": [], "data_sha256": None, "reference_sha256": _hash(target),
        "saved_path": None, "provenance_class": "reference-only",
        "claim_status": f"not-regenerated: {reason}"})

## Dataset preparation and optional long-run archive

No download is needed for the default run. The committed fixture records original sample indices, framing/resampling settings, upstream archive hashes, and software versions.

To download the full upstream datasets into `data/external_representations/` from inside this notebook environment:

```powershell
uv sync --extra repro --extra external
$env:MNN_ALLOW_DATA_DOWNLOADS = "1"
$env:MNN_DOWNLOAD_REPRESENTATION_DATA = "1"
```

Then rerun this notebook. Downloads never start unless both flags are set. Large raw datasets remain ignored by Git. A provenance-complete long-run feature archive may be placed at `data/results/representation_feature_archive.npz`; set `MNN_USE_REPRESENTATION_ARCHIVE=1` with `MNN_RUN_PROFILE=publication` to replay it. Reference rasters are visual targets only and cannot be exported as regenerated evidence.

In [ ]:
NOTEBOOK_NAME = "04_representations.ipynb"
FIXTURE = ROOT / "data" / "fixtures" / "representations_reduced_datasets.npz"
FEATURE_ARCHIVE = ROOT / "data" / "results" / "representation_feature_archive.npz"
REFERENCE_DIR = ROOT / "experiments" / "assets" / "references"
FULL_DATA_ROOT = ROOT / "data" / "external_representations"
REQUIRED_ARCHIVE_META = {"runner", "runner_code_sha256", "created_utc", "profile", "seeds",
                         "datasets", "preprocessing", "models", "checkpoints", "software"}

def download_source_datasets():
    """Explicit opt-in download of the genuine N-MNIST and MNE sample sources."""
    if not (ALLOW_DATA_DOWNLOADS and DOWNLOAD_REPRESENTATION_DATA):
        raise RuntimeError("Set MNN_ALLOW_DATA_DOWNLOADS=1 and MNN_DOWNLOAD_REPRESENTATION_DATA=1")
    try:
        import mne, tonic
        from tonic.transforms import ToFrame
    except ImportError as error:
        raise RuntimeError("Install the external extra: uv sync --extra repro --extra external") from error
    nmnist_root = FULL_DATA_ROOT / "nmnist"
    transform = ToFrame(sensor_size=tonic.datasets.NMNIST.sensor_size, n_time_bins=10)
    tonic.datasets.NMNIST(save_to=str(nmnist_root), train=True, transform=transform)
    tonic.datasets.NMNIST(save_to=str(nmnist_root), train=False, transform=transform)
    mne_root = FULL_DATA_ROOT / "mne"
    mne.datasets.sample.data_path(path=str(mne_root), download=True, verbose=True)
    return {"nmnist": nmnist_root, "mne": mne_root}

if DOWNLOAD_REPRESENTATION_DATA:
    DOWNLOADED_DATASETS = download_source_datasets()

def load_feature_archive(path=FEATURE_ARCHIVE):
    if not path.exists():
        return None
    z = np.load(path, allow_pickle=False)
    required = {"nmnist_features", "nmnist_labels", "eeg_features", "eeg_labels", "metadata"}
    if required - set(z.files):
        raise ValueError(f"representation archive is incomplete: {sorted(required-set(z.files))}")
    meta = json.loads(str(z["metadata"].item()))
    missing = REQUIRED_ARCHIVE_META - set(meta)
    if missing:
        raise ValueError(f"representation archive metadata missing: {sorted(missing)}")
    for key in ("runner_code_sha256", *meta["checkpoints"].values()):
        if len(key) != 64 or any(c not in "0123456789abcdef" for c in key.lower()):
            raise ValueError("invalid runner/checkpoint hash")
    result = {"metadata": meta}
    for task in ("nmnist", "eeg"):
        features = np.asarray(z[f"{task}_features"], dtype=np.float32)
        labels = np.asarray(z[f"{task}_labels"], dtype=np.int64)
        if features.ndim != 2 or labels.ndim != 1 or len(features) != len(labels) or not np.isfinite(features).all():
            raise ValueError(f"invalid {task} feature archive")
        if len(np.unique(labels)) != (10 if task == "nmnist" else 4):
            raise ValueError(f"incomplete classes in {task} feature archive")
        result[task] = {"features": features, "labels": labels}
    return result

## Live second-convolution feature extraction

The training and extraction sequence follows the authored workflow: train the spiking convolutional model, run the samples again in evaluation mode, average/sum the second-convolution spike tensor over time, flatten it, then project and probe those features.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.preprocessing import StandardScaler

def _fixture_data(profile):
    if not FIXTURE.exists():
        raise FileNotFoundError(f"Missing committed reduced fixture: {FIXTURE}")
    z = np.load(FIXTURE, allow_pickle=False)
    metadata = json.loads(str(z["metadata"].item()))
    data = {"metadata": metadata}
    budgets = {
        "smoke": {"nmnist_train": 8, "nmnist_test": 2, "eeg_train": 8, "eeg_test": 2,
                  "nmnist_epochs": 1, "eeg_epochs": 1},
        "reduced": {"nmnist_train": 40, "nmnist_test": 20, "eeg_train": 48, "eeg_test": 16,
                    "nmnist_epochs": 40, "eeg_epochs": 60},
    }[profile]
    for task, classes in (("nmnist", 10), ("eeg", 4)):
        x, y = z[f"{task}_x"], z[f"{task}_y"]
        train_idx, test_idx = [], []
        n_train, n_test = budgets[f"{task}_train"], budgets[f"{task}_test"]
        for label in range(classes):
            idx = np.flatnonzero(y == label)
            train_idx.extend(idx[:n_train]); test_idx.extend(idx[n_train:n_train+n_test])
        data[task] = {"train_x": x[train_idx], "train_y": y[train_idx],
                      "test_x": x[test_idx], "test_y": y[test_idx]}
    data["budget"] = budgets
    return data

def _loader(x, y, seed, batch_size, shuffle):
    from torch.utils.data import DataLoader, TensorDataset
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(torch.as_tensor(x), torch.as_tensor(y, dtype=torch.long)),
        batch_size=batch_size, shuffle=shuffle, generator=generator if shuffle else None,
        num_workers=0, pin_memory=(DEVICE_RESOLVED == "cuda"))

def _checkpoint_hash(net):
    buffer = io.BytesIO(); torch.save(net.state_dict(), buffer)
    return hashlib.sha256(buffer.getvalue()).hexdigest()

def _move(value):
    return value.to(DEVICE_RESOLVED, non_blocking=(DEVICE_RESOLVED == "cuda"))

def _train_nmnist(data, epochs, seed=0):
    from snntorch import surrogate
    from mnn_torch.models import TemporalMCSNN
    from mnn_torch.training import make_config, precompute_device_params
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    batch = 32
    train_x = (np.asarray(data["train_x"]) > 0).astype(np.float32)
    test_x = (np.asarray(data["test_x"]) > 0).astype(np.float32)
    train_loader = _loader(train_x, data["train_y"], seed, batch, True)
    test_loader = _loader(test_x, data["test_y"], seed, batch, False)
    config = make_config(precompute_device_params(), prob=0.0, homeo=False,
                         disturb_conductance=False, pf_frozen_variability=True)
    net = TemporalMCSNN(beta=.95, spike_grad=surrogate.fast_sigmoid(slope=25),
        num_steps=10, batch_size=batch, num_kernels=5, num_conv1=8, num_conv2=16,
        max_pooling=2, num_outputs=10, input_shape=(2,34,34),
        memristive_config=config).to(DEVICE_RESOLVED)
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = torch.nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        for frames, labels in train_loader:
            frames = _move(frames.float()).permute(1,0,2,3,4); labels = _move(labels)
            spikes, membrane, _ = net(frames)
            loss = sum(loss_fn(membrane[t], labels) for t in range(membrane.shape[0])) / membrane.shape[0]
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    features, labels_out = [], []
    net.eval()
    with torch.no_grad():
        for frames, labels in test_loader:
            frames = _move(frames.float()).permute(1,0,2,3,4)
            _, _, conv2_spikes = net(frames)
            features.append(conv2_spikes.sum(0).flatten(1).cpu().numpy())
            labels_out.append(labels.numpy())
    return {"features": np.concatenate(features), "labels": np.concatenate(labels_out),
            "checkpoint_sha256": _checkpoint_hash(net), "layer": "temporal_mcsnn.conv2_spikes"}

def _train_eeg(data, epochs, seed=0):
    from snntorch import surrogate
    from mnn_torch.models import MCSNN
    from mnn_torch.training import make_config, precompute_device_params
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    batch = 16
    train_x = np.asarray(data["train_x"], dtype=np.float32)[:,None]
    test_x = np.asarray(data["test_x"], dtype=np.float32)[:,None]
    train_loader = _loader(train_x, data["train_y"], seed, batch, True)
    all_x = np.concatenate([train_x, test_x]); all_y = np.concatenate([data["train_y"], data["test_y"]])
    all_loader = _loader(all_x, all_y, seed, batch, False)
    config = make_config(precompute_device_params(), prob=.1, homeo=True,
                         disturb_mode="fixed", disturb_conductance=True,
                         pf_frozen_variability=True)
    config["conv_ideal"] = True  # published dense-only PF mapping; conv feature extractor remains exact
    net = MCSNN(beta=.95, spike_grad=surrogate.fast_sigmoid(slope=25), num_steps=10,
        batch_size=batch, num_kernels=4, num_conv1=8, num_conv2=16, max_pooling=2,
        num_outputs=4, input_shape=(1,all_x.shape[2],all_x.shape[3]),
        memristive_config=config).to(DEVICE_RESOLVED)
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = torch.nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        for samples, labels in train_loader:
            samples = _move(samples.float()); labels = _move(labels)
            _, membrane, _ = net(samples)
            loss = sum(loss_fn(membrane[t], labels) for t in range(membrane.shape[0]))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    features, labels_out = [], []
    net.eval()
    with torch.no_grad():
        for samples, labels in all_loader:
            _, _, conv2_spikes = net(_move(samples.float()))
            features.append(conv2_spikes.mean(0).flatten(1).cpu().numpy())
            labels_out.append(labels.numpy())
    return {"features": np.concatenate(features), "labels": np.concatenate(labels_out),
            "checkpoint_sha256": _checkpoint_hash(net), "layer": "mcsnn.conv2_spikes"}

def run_live_representations(profile):
    started = time.time(); fixture = _fixture_data(profile); budget = fixture["budget"]
    nmnist = _train_nmnist(fixture["nmnist"], budget["nmnist_epochs"])
    eeg = _train_eeg(fixture["eeg"], budget["eeg_epochs"])
    metadata = {"runner": "mnn-representations-reduced-v1", "profile": profile,
        "seed": 0, "device": DEVICE_RESOLVED, "fixture_sha256": _hash(FIXTURE),
        "budget": budget, "checkpoints": {"nmnist": nmnist["checkpoint_sha256"],
        "eeg": eeg["checkpoint_sha256"]}, "software": {"python": platform.python_version(),
        "torch": torch.__version__}}
    return {"metadata": metadata, "nmnist": nmnist, "eeg": eeg,
            "wall_seconds": time.time()-started}

archive = load_feature_archive() if (USE_REPRESENTATION_ARCHIVE or RUN_PROFILE == "publication") else None
if USE_REPRESENTATION_ARCHIVE and archive is None:
    raise FileNotFoundError(f"MNN_USE_REPRESENTATION_ARCHIVE=1 but no compatible archive exists at {FEATURE_ARCHIVE}")
if RUN_PROFILE in {"reduced", "smoke"}:
    REPRESENTATION_RESULTS = run_live_representations(RUN_PROFILE)
    REPRESENTATION_SOURCE = REPRESENTATION_RESULTS
    REPRESENTATION_PROVENANCE = "live-reduced"; REPRESENTATION_CLAIM = "reduced-validation"
elif archive is not None:
    REPRESENTATION_RESULTS = archive; REPRESENTATION_SOURCE = FEATURE_ARCHIVE
    REPRESENTATION_PROVENANCE = "live-exact"; REPRESENTATION_CLAIM = "claimable-runner-archive"
else:
    REPRESENTATION_RESULTS = None; REPRESENTATION_SOURCE = None
    REPRESENTATION_PROVENANCE = "reference-only"; REPRESENTATION_CLAIM = "not-regenerated"

{"profile": RUN_PROFILE, "source": "live reduced samples" if REPRESENTATION_RESULTS and REPRESENTATION_PROVENANCE == "live-reduced" else "optional archive/reference",
 "wall_seconds": REPRESENTATION_RESULTS.get("wall_seconds") if REPRESENTATION_RESULTS else None,
 "feature_shapes": {key: REPRESENTATION_RESULTS[key]["features"].shape for key in ("nmnist","eeg")} if REPRESENTATION_RESULTS else None}

## N-MNIST class separability

PCA, t-SNE and UMAP follow the manuscript composite. PCA uses a 20-dimensional held-out linear probe while its left panel shows the two-dimensional projection. UMAP is fitted on the training fold and transformed out of sample. Because t-SNE has no out-of-sample transform, its stated probe is stratified five-fold cross-validation and remains visualization-grade.

In [ ]:
if REPRESENTATION_RESULTS is None:
    show_reference_only("fig_nmnist_separability.png", "publication archive not supplied")
else:
    try:
        import umap
    except ImportError as error:
        raise RuntimeError("UMAP is required for reproduction: install the repro extra") from error
    features = np.asarray(REPRESENTATION_RESULTS["nmnist"]["features"], float)
    labels = np.asarray(REPRESENTATION_RESULTS["nmnist"]["labels"], int)
    probe_test_size = max(len(np.unique(labels)), int(round(.20*len(labels))))
    idx_train, idx_test = train_test_split(np.arange(len(labels)), test_size=probe_test_size,
        stratify=labels, random_state=0)
    pca_probe = PCA(n_components=min(20, features.shape[1], len(idx_train)-1), random_state=0).fit(features[idx_train])
    classifier = LogisticRegression(max_iter=1000).fit(pca_probe.transform(features[idx_train]), labels[idx_train])
    pca_pred = classifier.predict(pca_probe.transform(features[idx_test]))
    pca_emb = PCA(n_components=2, random_state=0).fit_transform(features)
    perplexity = min(30, max(5, (len(features)-1)//3))
    tsne_emb = TSNE(n_components=2, perplexity=perplexity, learning_rate=200,
                    random_state=42, init="pca").fit_transform(features)
    folds = min(5, int(np.bincount(labels).min()))
    cv = StratifiedKFold(folds, shuffle=True, random_state=0)
    tsne_pred = cross_val_predict(LogisticRegression(max_iter=1000), tsne_emb, labels, cv=cv)
    umap_model = umap.UMAP(n_components=2, random_state=42, n_jobs=1).fit(features[idx_train])
    umap_train = umap_model.transform(features[idx_train]); umap_test = umap_model.transform(features[idx_test])
    classifier = LogisticRegression(max_iter=1000).fit(umap_train, labels[idx_train])
    umap_pred = classifier.predict(umap_test); umap_emb = umap_model.transform(features)
    rows = [("PCA", pca_emb, labels[idx_test], pca_pred, "held-out probe"),
            ("t-SNE", tsne_emb, labels, tsne_pred, f"{folds}-fold CV, viz-only"),
            ("UMAP", umap_emb, labels[idx_test], umap_pred, "held-out probe")]
    fig, axes = plt.subplots(3, 2, figsize=(8, 12))
    NMNIST_PROBE_ACCURACY = {}
    for row, (name, embedded, true, pred, protocol) in enumerate(rows):
        accuracy = 100*accuracy_score(true, pred); NMNIST_PROBE_ACCURACY[name] = accuracy
        axes[row,0].scatter(embedded[:,0], embedded[:,1], c=labels, cmap="tab10", s=9, alpha=.8)
        axes[row,0].set_title(f"{name} embedding (colour = digit class)")
        axes[row,0].set_ylabel(f"{accuracy:.1f}% ({protocol})")
        cm = confusion_matrix(true, pred, labels=np.arange(10), normalize="true")
        axes[row,1].imshow(cm, vmin=0, vmax=1, cmap="Blues")
        axes[row,1].set_title(f"{name} probe confusion")
        axes[row,1].set_xlabel("predicted"); axes[row,1].set_ylabel("true")
        axes[row,1].set_xticks(range(10)); axes[row,1].set_yticks(range(10))
    fig.tight_layout()
    finish(fig, "fig_nmnist_separability.png", REPRESENTATION_SOURCE,
        REFERENCE_DIR/"fig_nmnist_separability.png", seeds=[0],
        provenance_class=REPRESENTATION_PROVENANCE, claim_status=REPRESENTATION_CLAIM)
    NMNIST_PROBE_ACCURACY

## EEG representation composite

This reproduces the appendix layout directly: each two-dimensional embedding is overlaid with an in-sample logistic-regression boundary and paired with its confusion matrix. The in-sample score is a visualization diagnostic, not a held-out generalization claim.

In [ ]:
if REPRESENTATION_RESULTS is None:
    show_reference_only("fig6_eeg-embeddings.png", "publication archive not supplied")
else:
    import umap
    features = StandardScaler().fit_transform(np.asarray(REPRESENTATION_RESULTS["eeg"]["features"], float))
    labels = np.asarray(REPRESENTATION_RESULTS["eeg"]["labels"], int)
    label_map = {0:"auditory/left", 1:"auditory/right", 2:"visual/left", 3:"visual/right"}
    class_names = [label_map[i] for i in sorted(label_map)]
    perplexity = min(30, max(5, (len(features)-1)//3))
    reducers = [("PCA", PCA(n_components=2)),
                ("t-SNE", TSNE(n_components=2, perplexity=perplexity, learning_rate=200, random_state=42, init="pca")),
                ("UMAP", umap.UMAP(n_components=2, random_state=42, n_jobs=1))]
    fig, axes = plt.subplots(3, 2, figsize=(13, 15))
    EEG_VISUAL_ACCURACY = {}
    panel_labels = (("a)", "d)"), ("b)", "e)"), ("c)", "f)"))
    for row, (name, reducer) in enumerate(reducers):
        embedded = reducer.fit_transform(features)
        classifier = LogisticRegression(max_iter=1000).fit(embedded, labels)
        prediction = classifier.predict(embedded)
        EEG_VISUAL_ACCURACY[name] = 100*accuracy_score(labels, prediction)
        x0,x1 = embedded[:,0].min()-1, embedded[:,0].max()+1
        y0,y1 = embedded[:,1].min()-1, embedded[:,1].max()+1
        gx = np.linspace(x0,x1,240); gy = np.linspace(y0,y1,240); xx,yy = np.meshgrid(gx,gy)
        zz = classifier.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
        axes[row,0].contourf(xx,yy,zz,alpha=.20,cmap="coolwarm")
        for label in range(4):
            mask = labels == label
            axes[row,0].scatter(embedded[mask,0], embedded[mask,1], s=32,
                edgecolors="white", linewidths=.3, label=label_map[label])
        axes[row,0].set_title(f"{name} Embedding with Logistic Regression")
        axes[row,0].legend(title="Class", fontsize=8)
        axes[row,0].text(-.07,1.04,panel_labels[row][0],transform=axes[row,0].transAxes,
                         fontsize=15,fontweight="bold")
        cm = confusion_matrix(labels, prediction, labels=np.arange(4))
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
            ax=axes[row,1], cmap="Blues", colorbar=False, xticks_rotation=45)
        axes[row,1].set_title(f"{name} Confusion Matrix")
        axes[row,1].text(-.36,1.04,panel_labels[row][1],transform=axes[row,1].transAxes,
                         fontsize=15,fontweight="bold")
    fig.tight_layout()
    finish(fig, "fig6_eeg-embeddings.png", REPRESENTATION_SOURCE,
        REFERENCE_DIR/"fig6_eeg-embeddings.png", seeds=[0],
        provenance_class=REPRESENTATION_PROVENANCE, claim_status=REPRESENTATION_CLAIM)
    EEG_VISUAL_ACCURACY

## Reproduction report

In [ ]:
expected = [x for x in MANIFEST["figures"] if x["notebook"] == NOTEBOOK_NAME]
assert len(FIGURE_REPORT) == len(expected)
{"figures": FIGURE_REPORT,
 "validation_metrics": {"nmnist_probe_accuracy": globals().get("NMNIST_PROBE_ACCURACY"),
                        "eeg_in_sample_visual_accuracy": globals().get("EEG_VISUAL_ACCURACY")}}